# Crystal Studio — live demo  (free-energy-penalty model, physics_weight = 0.02)

A walkthrough of the interactive tool's workflow, using the existing
**`fno_grains_tuned.pt`** model (trained with the free-energy dissipation
penalty at 0.02). For each seed we:

1. **seed** → build a PFC initial field (a drawn shape, or a preset seed),
2. **predict** → grow the crystal with the FNO (one forward pass per frame),
3. **measure** → report grain count, orientation, defects, lattice wavelength,
   crystallinity,
4. **check** → compare against a target spec the user wants,

plus a physics-consistency **confidence** score for the prediction.

> This demonstrates the intended tool. Arbitrary freehand seeds are the next
> step (a `custom_mask` fine-tune); here we use seeds close to the model's
> training distribution so the demo is faithful.

Set **Runtime → T4 GPU**, then **Run all**.

### Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy matplotlib pyyaml
import sys; sys.path.insert(0, 'crystal_tool')

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
import grow as G, analyze as A, check as C
from pfc_solver import PFCConfig, make_initial_condition

CKPT = '/content/drive/MyDrive/fno_grains_tuned.pt'    # the physics_weight=0.02 model
import os; assert os.path.exists(CKPT), f"missing checkpoint: {CKPT} (change CKPT)"
model, ck, device = G.load_fno(CKPT)
pw = ck['config']['train'].get('physics_weight')
print(f"loaded {CKPT}")
print(f"  physics_weight = {pw}   (free-energy dissipation penalty)")
print(f"  trained to epoch {ck.get('epoch')}, val {ck.get('val_loss'):.3e}")
print(f"  device = {device}")
STEPS = 30

### The demo function
Builds the seed, grows it, measures it, checks it against the target, and draws
one summary panel: the growth filmstrip, the final crystal with detected atoms
(cyan) and defects (green), the measured characteristics, and the target check.

In [ ]:
def make_disk(N, cx, cy, rad):
    Y, X = np.mgrid[0:N, 0:N]
    return (np.sqrt((X-cx)**2 + (Y-cy)**2) < rad).astype(float)

def run_demo(title, seed, target):
    # --- seed -> initial field ---
    if seed['kind'] == 'mask':
        field0, cfg = G.mask_to_field(seed['mask'], r=seed['r'], n0=seed['n0'], N=128)
    else:                                    # preset solver seed (e.g. multi)
        cfg = PFCConfig(r=seed['r'], n0=seed['n0'], seed_type=seed['seed_type'],
                        n_seeds=seed.get('n_seeds', 12), seed_k0=1.0, rng_seed=7)
        field0 = make_initial_condition(cfg, np.random.default_rng(7)).astype('float32')
    # --- predict ---
    frames = G.grow_fno(model, ck, device, field0, steps=STEPS)
    props = A.analyze(frames[-1], cfg.dx)
    rows, ok = C.check(props, target)
    _, trust, _ = G.pde_confidence(frames, cfg, device='cpu')

    # --- one summary panel ---
    fig = plt.figure(figsize=(15, 5.2)); gs = fig.add_gridspec(2, 6, height_ratios=[1, 1.15])
    idx = np.linspace(0, len(frames) - 1, 6).astype(int)
    for k, fi in enumerate(idx):
        ax = fig.add_subplot(gs[0, k]); ax.imshow(frames[fi], cmap='magma'); ax.axis('off')
        ax.set_title(f"frame {fi}", fontsize=9)
    axf = fig.add_subplot(gs[1, 0:2]); axf.imshow(frames[-1], cmap='magma')
    if len(props['atoms']):
        axf.scatter(props['atoms'][:, 1], props['atoms'][:, 0], s=4, c='cyan', alpha=0.5)
    if len(props['defect_coords']):
        axf.scatter(props['defect_coords'][:, 1], props['defect_coords'][:, 0],
                    s=45, facecolors='none', edgecolors='lime', linewidths=1.3)
    axf.set_title('final crystal — atoms (cyan), defects (green)', fontsize=9); axf.axis('off')
    axt = fig.add_subplot(gs[1, 2:4]); axt.axis('off')
    axt.text(0, 1, "MEASURED\n" + "\n".join(A.summary_lines(props)),
             va='top', family='monospace', fontsize=10)
    axc = fig.add_subplot(gs[1, 4:6]); axc.axis('off')
    lines = ["TARGET CHECK"]
    for r in rows:
        lines.append(f"  [{'OK ' if r['ok'] else 'NO '}] {r['criterion']}: "
                     f"want {r['desired']}  got {r['actual']}")
    lines.append(f"\n  MATCH: {'YES ✓' if ok else 'NO ✗'}")
    lines.append(f"  prediction confidence: {trust*100:.0f}%")
    axc.text(0, 1, "\n".join(lines), va='top', family='monospace', fontsize=10,
             color='darkgreen' if ok else 'firebrick')
    fig.suptitle(title, fontsize=14, fontweight='bold'); fig.tight_layout(); plt.show()
    return props

## Demo 1 — draw a blob → a single crystal grows
Target: a clean single crystal.

In [ ]:
mask = make_disk(128, 64, 64, 16)
_ = run_demo("Demo 1 · drawn blob → single crystal",
             {'kind': 'mask', 'mask': mask, 'r': -0.30, 'n0': -0.285},
             {'structure': 'single', 'max_defects': 10, 'min_crystallinity': 0.7})

## Demo 2 — a multi-seed → a polycrystal
Target: a polycrystal (the analyzer should find grains + boundary defects).

In [ ]:
_ = run_demo("Demo 2 · multi-seed → polycrystal",
             {'kind': 'preset', 'seed_type': 'multi', 'n_seeds': 14, 'r': -0.30, 'n0': -0.285},
             {'structure': 'poly', 'min_crystallinity': 0.6})

## Demo 3 — draw two blobs → check a custom target
Target the user might want: single crystal, very few defects (will it match?).

In [ ]:
mask = np.maximum(make_disk(128, 44, 50, 13), make_disk(128, 84, 80, 13))
_ = run_demo("Demo 3 · drawn shape vs a strict target",
             {'kind': 'mask', 'mask': mask, 'r': -0.33, 'n0': -0.285},
             {'structure': 'single', 'max_defects': 2, 'min_crystallinity': 0.8})

## What this shows — and what's next

- The full **draw → predict → measure → check** loop, running on the
  free-energy-penalty model, in real time (each frame is one FNO pass).
- Physical characteristics are computed automatically and compared to a target.
- A confidence score flags when the prediction drifts off the physics.

**Next steps shown to the meeting:** (1) a freehand drawing canvas (Streamlit),
(2) a `custom_mask` fine-tune so *arbitrary* drawn seeds predict accurately,
(3) "verify with the real solver" for ground truth, and eventually inverse
design — solving for the seed that yields a desired crystal.